# 🏛️ Notebook 3: A Central Config Server (with hot reload)

So far each service reads its *own* env vars or config file. In a real
microservices system you often have **dozens** of services that need to
share settings (timeouts, feature flags, URLs, limits).

The **Config Server pattern**:

```
┌────────────┐    GET /config/orders     ┌────────────────┐
│ orders svc │ ────────────────────────► │ config server  │
│ pay svc    │ ────────────────────────► │  (central DB)  │
│ ship svc   │ ────────────────────────► │                │
└────────────┘                           └────────────────┘
```

Real examples: **Spring Cloud Config**, **HashiCorp Consul**,
**etcd**, **AWS AppConfig**, **Kubernetes ConfigMaps**.


## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 A mini config server

We'll simulate a central server with a plain Python class and have two
"services" pull from it. No network — just to show the shape of the pattern.


In [1]:
import time, threading

class ConfigServer:
    """Central store. Versioned so clients can tell when config changes."""
    def __init__(self):
        self._data: dict[str, dict] = {}
        self._version = 0
        self._lock = threading.Lock()

    def put(self, service: str, cfg: dict):
        with self._lock:
            self._data[service] = dict(cfg)
            self._version += 1
            print(f"[server] {service} updated -> v{self._version}")

    def get(self, service: str) -> tuple[int, dict]:
        with self._lock:
            return self._version, dict(self._data.get(service, {}))

server = ConfigServer()
server.put("orders", {"timeout_ms": 500, "retries": 3})
server.put("payments", {"timeout_ms": 2000, "provider": "stripe"})
print(server.get("orders"))
print(server.get("payments"))


[server] orders updated -> v1
[server] payments updated -> v2
(2, {'timeout_ms': 500, 'retries': 3})
(2, {'timeout_ms': 2000, 'provider': 'stripe'})


## 🔄 Hot reload from the client side

A service **polls** the server (or subscribes) and updates its in-memory
config when the version changes. No restart needed.


In [2]:
class ServiceClient:
    def __init__(self, name: str, server: ConfigServer):
        self.name = name
        self.server = server
        self.version = -1
        self.config: dict = {}
        self.refresh()

    def refresh(self):
        v, cfg = self.server.get(self.name)
        if v != self.version:
            self.version = v
            self.config = cfg
            print(f"[{self.name}] reloaded config v{v}: {cfg}")

    def handle_request(self, req: str) -> str:
        self.refresh()  # a real app might poll every N seconds instead
        return f"{self.name} handled {req!r} with timeout={self.config.get('timeout_ms')}ms"

orders = ServiceClient("orders", server)
print(orders.handle_request("buy"))

# Ops bumps the timeout live — no redeploy:
server.put("orders", {"timeout_ms": 1500, "retries": 5})
print(orders.handle_request("buy"))


[orders] reloaded config v2: {'timeout_ms': 500, 'retries': 3}
orders handled 'buy' with timeout=500ms
[server] orders updated -> v3
[orders] reloaded config v3: {'timeout_ms': 1500, 'retries': 5}
orders handled 'buy' with timeout=1500ms


## 📣 Push-based: watchers / subscribers

Polling is simple but either laggy (long interval) or wasteful (short
interval). Real tools like **etcd**, **Consul**, and **ZooKeeper** let
clients **subscribe** — the server pushes only when something changes.

Here's the same pattern with plain callbacks.


In [3]:
class PushConfigServer(ConfigServer):
    def __init__(self):
        super().__init__()
        self._subscribers: dict[str, list] = {}

    def subscribe(self, service: str, callback):
        self._subscribers.setdefault(service, []).append(callback)

    def put(self, service: str, cfg: dict):
        super().put(service, cfg)
        for cb in self._subscribers.get(service, []):
            cb(self._version, dict(cfg))

push_server = PushConfigServer()

class PushClient:
    def __init__(self, name, server):
        self.name, self.server = name, server
        self.config: dict = {}
        server.subscribe(name, self._on_change)

    def _on_change(self, version, cfg):
        self.config = cfg
        print(f"[{self.name}] pushed v{version}: {cfg}")

orders = PushClient("orders", push_server)
push_server.put("orders", {"timeout_ms": 500})
push_server.put("orders", {"timeout_ms": 2000})   # arrives instantly, no polling


[server] orders updated -> v1
[orders] pushed v1: {'timeout_ms': 500}
[server] orders updated -> v2
[orders] pushed v2: {'timeout_ms': 2000}


### 🔁 Pull vs push — which to use?

| | Pull (poll) | Push (subscribe) |
|--|--|--|
| Client complexity | very low | needs a persistent connection |
| Latency to change | up to the poll interval | ~instant |
| Server load | constant | proportional to changes |
| Works behind strict firewalls | yes (outbound HTTP) | sometimes harder |

Most real systems combine them: **long-poll / SSE / gRPC streams** so the
client still initiates the connection (firewall-friendly) but the server
only replies when something changes.


## 🛟 What happens if the config server is down?

Rule: **a config outage must not take your services down.**
Clients should cache the last-known-good config and keep serving traffic.


In [4]:
class ResilientClient(ServiceClient):
    def refresh(self):
        try:
            v, cfg = self.server.get(self.name)
            if v != self.version:
                self.version = v
                self.config = cfg
                print(f"[{self.name}] reloaded v{v}")
        except Exception as e:                  # e.g. network error
            print(f"[{self.name}] config server down ({e}); using cached v{self.version}")

# Break the server and make sure the client keeps working.
class BrokenServer:
    def get(self, service): raise ConnectionError("config server unreachable")

rc = ResilientClient("orders", server)   # fetch good config first
rc.server = BrokenServer()                # now simulate outage
print(rc.handle_request("buy"))           # still uses cached config


[orders] reloaded v3
[orders] config server down (config server unreachable); using cached v3
orders handled 'buy' with timeout=1500ms


## 🏗️ Real-world shapes of this pattern

| Tool | Style | Notes |
|---|---|---|
| **Spring Cloud Config** | HTTP + git backend | Classic JVM microservices setup |
| **Consul / etcd** | KV store + watch | Used as a primitive by many systems |
| **Kubernetes ConfigMap/Secret** | Mounted as files or env | Restart pod *or* use a sidecar reloader |
| **AWS AppConfig / Parameter Store** | Managed service | Validation + staged rollout built-in |
| **LaunchDarkly / Unleash** | Flags-focused | Great for per-user targeting (see NB 2) |

## 🧠 Takeaways

- Centralising config scales *much* better than per-service env vars.
- Always **version** config so clients know when to reload.
- Clients must tolerate the server being **temporarily unreachable**.
- Combine with feature flags (NB 2) and secrets (NB 4) for a complete picture.
